In [ ]:
# ============================================================
# LEARNING CURVE
# ============================================================
import os, json, random, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
warnings.filterwarnings("ignore")

# -------------------- CONFIG --------------------
FID          = 0
FRACTIONS    = [0.2, 0.4, 0.6, 0.8, 1.0]
SEEDS        = [42, 123, 2024]          
EPOCHS       = 15                       
BATCH_SIZE   = 32
LR, WD, WARMUP = 2e-4, 1e-4, 2
LABEL_SMOOTH = 0.0                      
USE_CLASS_W  = True                     
PRETRAINED   = True                    
LC_DIR       = "/kaggle/working/output/learning_curve_v2"
os.makedirs(LC_DIR, exist_ok=True)
RUNS_PATH    = os.path.join(LC_DIR, "runs.json")

tr_full = np.array(folds[FID]["train"])
va_idx  = np.array(folds[FID]["val"])
te_idx  = np.array(folds[FID]["test"])
assert len(set(tr_full) & set(te_idx)) == 0 and len(set(va_idx) & set(te_idx)) == 0, "Leakage!"
print(f"Fold {FID} | Train(full) {len(tr_full)} | Val {len(va_idx)} | Test {len(te_idx)}")

TRAIN_TF = globals().get("train_tf", eval_tf)
if "train_tf" not in globals():
    print("⚠️ train_tf nai, eval_tf (no augmentation) diye train hobe.")

# -------------------- HELPERS --------------------
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def mk_loader(idx, tf, shuffle):
    return DataLoader(DentalDS(X, y, idx, tf), batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=0, pin_memory=True, drop_last=shuffle)

@torch.no_grad()
def eval_split(model, loader):
    """Plain CE loss + acc + macro-F1 (no smoothing, no class weight, no augmentation)."""
    model.eval()
    tot_loss, n, P, Yl = 0.0, 0, [], []
    for xb, yb in loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=use_amp):
            out = model(xb)
        out = out.float()
        tot_loss += F.cross_entropy(out, yb, reduction="sum").item()
        n += len(yb)
        P.append(out.argmax(1).cpu()); Yl.append(yb.cpu())
    P, Yl = torch.cat(P).numpy(), torch.cat(Yl).numpy()
    return {"loss": tot_loss / n, "acc": accuracy_score(Yl, P),
            "f1": f1_score(Yl, P, average="macro")}

def subsample(frac, seed):
    if frac >= 1.0:
        return tr_full
    idx, _ = train_test_split(tr_full, train_size=frac, stratify=y[tr_full], random_state=seed)
    return np.sort(idx)

def load_runs():
    try:
        if os.path.exists(RUNS_PATH) and os.path.getsize(RUNS_PATH) > 0:
            return json.load(open(RUNS_PATH))
    except Exception as e:
        print("⚠️ runs.json read failed:", e)
    return []

def save_runs(runs):
    tmp = RUNS_PATH + ".tmp"
    json.dump(runs, open(tmp, "w"), indent=1)
    os.replace(tmp, RUNS_PATH)

va_loader = mk_loader(va_idx, eval_tf, False)
te_loader = mk_loader(te_idx, eval_tf, False)

# -------------------- ONE RUN --------------------
def run_once(frac, seed):
    seed_everything(seed)
    tr_idx = subsample(frac, seed)
    tr_loader      = mk_loader(tr_idx, TRAIN_TF, True)     # training (augmented)
    tr_eval_loader = mk_loader(tr_idx, eval_tf, False)     # train metric (no aug)

    if USE_CLASS_W:
        cnt = np.bincount(y[tr_idx], minlength=len(CLASSES))
        w = torch.tensor(len(tr_idx) / (len(CLASSES) * cnt + 1e-6), dtype=torch.float32).to(device)
    else:
        w = None
    criterion = nn.CrossEntropyLoss(weight=w, label_smoothing=LABEL_SMOOTH)

    model = build_model(pretrained=PRETRAINED).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.SequentialLR(opt, [
        torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1, total_iters=WARMUP),
        torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(EPOCHS - WARMUP, 1))
    ], milestones=[WARMUP])
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    best_f1, best_state = -1.0, None
    for ep in range(EPOCHS):
        model.train()
        for xb, yb in tr_loader:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                loss = criterion(model(xb).float(), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
        sched.step()
        v = eval_split(model, va_loader)
        if v["f1"] > best_f1:
            best_f1 = v["f1"]
            best_state = {k: t.detach().clone() for k, t in model.state_dict().items()}

    model.load_state_dict(best_state)
    tr = eval_split(model, tr_eval_loader)
    va = eval_split(model, va_loader)
    te = eval_split(model, te_loader)
    res = {"frac": frac, "seed": seed, "n_train": int(len(tr_idx)),
           "train_acc": tr["acc"] * 100, "val_acc": va["acc"] * 100,
           "train_loss": tr["loss"], "val_loss": va["loss"],
           "train_f1": tr["f1"] * 100, "val_f1": va["f1"] * 100,
           "test_acc": te["acc"] * 100, "test_f1": te["f1"] * 100}
    del model, opt, best_state; torch.cuda.empty_cache()
    return res

# -------------------- RUN ALL (resume-safe) --------------------
runs = load_runs()
done = {(r["frac"], r["seed"]) for r in runs}
print(f"Total runs {len(FRACTIONS) * len(SEEDS)} | already done {len(done)}\n")

for frac in FRACTIONS:
    for seed in SEEDS:
        if (frac, seed) in done:
            print(f"⏭️  Skip frac={frac} seed={seed}"); continue
        t0 = time.time()
        try:
            r = run_once(frac, seed)
        except Exception as e:
            print(f"❌ frac={frac} seed={seed} failed -> {e}")
            torch.cuda.empty_cache(); continue
        runs.append(r); save_runs(runs)
        print(f"✅ frac={frac} seed={seed} | n={r['n_train']} | TrAcc {r['train_acc']:.2f} | "
              f"VaAcc {r['val_acc']:.2f} | TrLoss {r['train_loss']:.4f} | VaLoss {r['val_loss']:.4f} | {time.time()-t0:.0f}s")

# -------------------- AGGREGATE --------------------
df = pd.DataFrame(runs); assert len(df) > 0, "Kono run complete hoy nai"
sd = lambda s: s.std(ddof=1)
agg = df.groupby("frac").agg(
    n_train=("n_train", "first"), reps=("seed", "count"),
    tra_m=("train_acc", "mean"), tra_s=("train_acc", sd),
    vaa_m=("val_acc", "mean"),   vaa_s=("val_acc", sd),
    trl_m=("train_loss", "mean"), trl_s=("train_loss", sd),
    val_m=("val_loss", "mean"),   val_s=("val_loss", sd),
    vaf_m=("val_f1", "mean"),     vaf_s=("val_f1", sd),
    tef_m=("test_f1", "mean"),    tef_s=("test_f1", sd),
    tea_m=("test_acc", "mean"),   tea_s=("test_acc", sd),
).reset_index().sort_values("frac")
agg = agg.fillna(0.0)

pm = lambda m, s, d=2: [f"{a:.{d}f} ± {b:.{d}f}" for a, b in zip(m, s)]
table1 = pd.DataFrame({
    "Fraction": agg["frac"],
    "Train Acc (%)": pm(agg.tra_m, agg.tra_s),
    "Val Acc (%)":   pm(agg.vaa_m, agg.vaa_s),
    "Train Loss":    pm(agg.trl_m, agg.trl_s, 4),
    "Val Loss":      pm(agg.val_m, agg.val_s, 4),
})
table2 = pd.DataFrame({
    "Fraction": agg["frac"], "N train": agg["n_train"],
    "Val Macro-F1 (%)":  pm(agg.vaf_m, agg.vaf_s),
    "Test Acc (%)":      pm(agg.tea_m, agg.tea_s),
    "Test Macro-F1 (%)": pm(agg.tef_m, agg.tef_s),
})
print("\n" + "=" * 90)
print(f"Table 1. Learning curve results across training data fractions (mean ± std over {len(SEEDS)} repeats)")
print("=" * 90)
print(table1.to_string(index=False))
print("\nSupplementary (Macro-F1 / Test)")
print(table2.to_string(index=False))
table1.to_csv(os.path.join(LC_DIR, "table1_learning_curve.csv"), index=False)
table2.to_csv(os.path.join(LC_DIR, "table2_macroF1_test.csv"), index=False)
df.to_csv(os.path.join(LC_DIR, "all_runs.csv"), index=False)

# -------------------- FIGURE 1: accuracy (tomar style) --------------------
x = agg["frac"].values
fig, ax = plt.subplots(figsize=(6.4, 4.6))
for m, s, col, lab in [("tra_m", "tra_s", "blue", "Training accuracy"),
                       ("vaa_m", "vaa_s", "red",  "Validation accuracy")]:
    mu, sdv = agg[m].values / 100, agg[s].values / 100
    ax.plot(x, mu, "o-", color=col, lw=2, label=lab)
    ax.fill_between(x, mu - sdv, mu + sdv, color=col, alpha=0.15)
ax.set_xlabel("Fraction of Training Data Used"); ax.set_ylabel("Accuracy")
ax.grid(alpha=0.3); ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(LC_DIR, "learning_curve_accuracy.png"), dpi=600)
plt.show()

# -------------------- FIGURE 2: loss + Macro-F1 --------------------
fig, ax = plt.subplots(1, 2, figsize=(11, 4.3))
for m, s, col, lab in [("trl_m", "trl_s", "blue", "Training loss"),
                       ("val_m", "val_s", "red",  "Validation loss")]:
    mu, sdv = agg[m].values, agg[s].values
    ax[0].plot(x, mu, "o-", color=col, lw=2, label=lab)
    ax[0].fill_between(x, mu - sdv, mu + sdv, color=col, alpha=0.15)
ax[0].set_xlabel("Fraction of Training Data Used"); ax[0].set_ylabel("Cross-entropy loss")
ax[0].set_title("Loss"); ax[0].grid(alpha=0.3); ax[0].legend()

mu, sdv = agg["vaf_m"].values, agg["vaf_s"].values
ax[1].plot(x, mu, "o-", color="green", lw=2, label="Validation Macro-F1")
ax[1].fill_between(x, mu - sdv, mu + sdv, color="green", alpha=0.15)
ax[1].set_xlabel("Fraction of Training Data Used"); ax[1].set_ylabel("Macro-F1 (%)")
ax[1].set_title("Macro-F1"); ax[1].grid(alpha=0.3); ax[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(LC_DIR, "learning_curve_loss_f1.png"), dpi=600)
plt.show()

print(f"\n✅ Done. Saved to {LC_DIR}")